# Video Super-Resolution: Analysis & Comparison

Load results from all models and generate comparison tables, plots, and visualizations.

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.utils import visualize_comparison, tensor_to_numpy

## Load Results

In [ ]:
models = ["baseline", "temporal_loss", "multiframe", "recurrent", "flow_warp"]
results = {}

for name in models:
    path = os.path.join("..", "results", name, "metrics.json")
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
        print(f"Loaded {name}")
    else:
        print(f"Missing results for {name}")

print(f"\nLoaded {len(results)}/{len(models)} models")

## Quantitative Comparison Table

In [ ]:
print(f"{'Model':<20} {'PSNR (dB)':>12} {'SSIM':>12} {'Temp. Consist.':>16}")
print("-" * 62)
for name in models:
    if name in results:
        r = results[name]
        print(
            f"{name:<20} "
            f"{r['psnr_mean']:>8.2f} ± {r['psnr_std']:.2f} "
            f"{r['ssim_mean']:>8.4f} ± {r['ssim_std']:.4f} "
            f"{r['temporal_consistency']:>12.4f}"
        )

## PSNR & SSIM Bar Charts

In [ ]:
available = [m for m in models if m in results]
psnr_vals = [results[m]["psnr_mean"] for m in available]
ssim_vals = [results[m]["ssim_mean"] for m in available]
tc_vals = [results[m]["temporal_consistency"] for m in available]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(available, psnr_vals, color="steelblue")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR Comparison")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(available, ssim_vals, color="coral")
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM Comparison")
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(available, tc_vals, color="seagreen")
axes[2].set_ylabel("Temporal Error (lower = better)")
axes[2].set_title("Temporal Consistency")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../results/comparison_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## Visual Comparison (Side-by-Side Frames)

In [ ]:
# Load comparison images from each model's results
from PIL import Image

fig, axes = plt.subplots(len(available), 1, figsize=(15, 5 * len(available)))
if len(available) == 1:
    axes = [axes]

for i, name in enumerate(available):
    img_path = os.path.join("..", "results", name, "seq_000", "comparison.png")
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(name)
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("../results/all_comparisons.png", dpi=150, bbox_inches="tight")
plt.show()

## Temporal Profile Visualization

Take a vertical slice through time to visualize temporal consistency.

In [ ]:
# This cell requires saved SR frame sequences
# For each model, load a sequence and extract a vertical column across time

print("To generate temporal profiles, run evaluate.py for each model first.")
print("Then extend this cell to load SR frames and create xt-slices.")